In [ ]:

USER_QUERY= "get details from Products with ID = 2 and tell me whether it is discontinued"

In [ ]:
import requests as r
from utils import query_raven



def get_northwind_data_details(Category: str, ID: str):
    """
    Get Categories. You can collect Order with its Order ID or Product with its Product ID.
    """
    url = f"https://services.odata.org/V2/Northwind/Northwind.svc/{Category}({ID})?$format=json"
    response = r.get(url)
    return response.json()




raven_functions = \
f'''
def get_northwind_data_details(Category : str,  ID : str):
    """
    
    Get Categories. You can collect Order with its Order ID or Product with its Product ID.

    """

User Query: {USER_QUERY}<human_end>
'''
call = query_raven(raven_functions)

# output['d']['ShippedDate']


In [ ]:
print(call)

In [ ]:

output = eval(call)
print(output)

In [ ]:
import os
from dotenv import load_dotenv  

# Load environment variables from .env file
load_dotenv()
# Access the API key using the variable name defined in the .env file
api_key = os.getenv("OPENAI_API_KEY")
# api_key = 'sk-proj-eh7lc5VDVDlkfSJ0x4QoT3BlbkFJxxlBBbiYyGLlEfRqzu8M'

In [ ]:
print(api_key)

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-3.5-turbo-1106", temperature=0.2)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content=USER_QUERY),
    HumanMessage(content=str(output)),
]

result=model.invoke(messages)

In [ ]:
print(messages)

In [ ]:
print(result.content)

<H1>Moving to more complicated function call scenarios</H1>


In [ ]:
USER_QUERY= "get details from Products with ID = 2 and tell me whether it is discontinued"

In [ ]:
import requests as r
from utils import query_raven



def get_top_selling_products(ProductID: list, Quantity: list):
    """
    Get ProductID and Quantity as separate lists from thei URL.
    """
    url = f"https://services.odata.org/V2/Northwind/Northwind.svc/Order_Details?$format=json&$select=Quantity,ProductID"
    response = r.get(url)
    return response.json()




raven_functions = \
f'''
def get_top_selling_products(ProductID: list, Quantity: list):
    """
    
    Get ProductID and Quantity as separate lists from thei URL.

    Parameters:
    Save the individual values of ProductID in the list ProductID
    Save the individual values of Quantity in the list Quantity


    """

User Query: {USER_QUERY}<human_end>
'''
call = query_raven(raven_functions)

# output['d']['ShippedDate']

In [ ]:
print(call)

In [ ]:
output = eval(call)
print(output)

In [ ]:
response2 = r.get('https://services.odata.org/V2/Northwind/Northwind.svc/Order_Details?$format=json&$select=Quantity,ProductID')

In [ ]:
response3=response2.json()

In [ ]:
raven_functions = \
f'''
def get_top_selling_products(ProductID: list, Quantity: list):
    """
    
    Get ProductID and Quantity as separate lists from thei URL.

    Parameters:
    using ProductID data in {str(response3)} Save the individual values of ProductID in the list ProductID
    using Quantity data in {str(response3)}Save the individual values of Quantity in the list Qunatity


    """

User Query: {USER_QUERY}<human_end>
'''
call = query_raven(raven_functions)

In [ ]:
print(call)

In [ ]:
response3

In [ ]:
import os
from dotenv import load_dotenv  
from langchain_openai import ChatOpenAI

# Load environment variables from .env file
load_dotenv()
# Access the API key using the variable name defined in the .env file
api_key = os.getenv("OPENAI_API_KEY")
model = ChatOpenAI(model="gpt-3.5-turbo")
# model = ChatOpenAI(model="gpt-4")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    # SystemMessage(content="compare the 'RequiredDate' and 'ShippedDate' for each order and tell me many  orders have been delayed in their delivery?"),
    SystemMessage(content="List top 3 products with highest quantity of sales"),
    HumanMessage(content=str(response3)),
]

result=model.invoke(messages)
print(result.content)

In [ ]:
import requests

# API URL
url = "https://services.odata.org/V2/Northwind/Northwind.svc/Order_Details?$format=json&$select=Quantity,ProductID"

# Fetch the data from the API
response = requests.get(url)
data = response.json()

# Print the entire JSON response to understand its structure
# print(data)

# Check if 'd' key is present and contains 'results'
if 'd' in data and 'results' in data['d']:
    results = data['d']['results']
    
    # Extract ProductID and Quantity into separate lists
    product_ids = [item['ProductID'] for item in results]
    quantities = [item['Quantity'] for item in results]

    # Print the lists
    print("Product IDs:", product_ids)
    print("Quantities:", quantities)
else:
    print("Unexpected JSON structure:", data)



In [ ]:
len(quantities)

In [ ]:
import requests
import matplotlib.pyplot as plt

# API URL
url = "https://services.odata.org/V2/Northwind/Northwind.svc/Order_Details?$format=json&$select=Quantity,ProductID"

# Fetch the data from the API
response = requests.get(url)
data = response.json()

# Check if 'd' key is present and contains 'results'
if 'd' in data and 'results' in data['d']:
    results = data['d']['results']
    
    # Extract ProductID and Quantity into separate lists
    product_ids = [item['ProductID'] for item in results]
    quantities = [item['Quantity'] for item in results]

    # Aggregate quantities by product ID
    product_quantity_dict = {}
    for pid, qty in zip(product_ids, quantities):
        if pid in product_quantity_dict:
            product_quantity_dict[pid] += qty
        else:
            product_quantity_dict[pid] = qty

    # Prepare data for plotting
    product_ids = list(product_quantity_dict.keys())
    total_quantities = list(product_quantity_dict.values())

    # Plotting
    plt.figure(figsize=(12, 6))
    plt.bar(product_ids, total_quantities, color='blue')
    plt.xlabel('Product ID')
    plt.ylabel('Total Quantity')
    plt.title('Total Quantity per Product ID')
    plt.xticks(rotation=90)
    plt.show()
else:
    print("Unexpected JSON structure:", data)


In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Function to convert OData date format to datetime
def parse_odata_date(date_str):
    # Extract the timestamp from the format /Date(836438400000)/
    timestamp = int(date_str[6:-2])
    return datetime.utcfromtimestamp(timestamp / 1000)

# API URL for Orders data
url = "https://services.odata.org/V2/Northwind/Northwind.svc/Orders?$format=json&$select=OrderID,OrderDate,ShippedDate"

# Fetch the data from the API
response = requests.get(url)
data = response.json()

# Check if 'd' key is present and contains 'results'
if 'd' in data and 'results' in data['d']:
    results = data['d']['results']
    
    # Create a DataFrame from the results
    df = pd.DataFrame(results)
    
    # Convert OrderDate and ShippedDate to datetime
    df['OrderDate'] = df['OrderDate'].apply(parse_odata_date)
    df['ShippedDate'] = df['ShippedDate'].apply(parse_odata_date)

    # Calculate fulfillment time in days
    df['FulfillmentTime'] = (df['ShippedDate'] - df['OrderDate']).dt.days

    # Drop rows with NaN values in FulfillmentTime (in case of missing dates)
    df = df.dropna(subset=['FulfillmentTime'])

    # Plotting the box plot
    plt.figure(figsize=(10, 6))
    plt.boxplot(df['FulfillmentTime'], vert=False, patch_artist=True, boxprops=dict(facecolor='lightblue'))
    plt.xlabel('Fulfillment Time (days)')
    plt.title('Distribution of Order Fulfillment Time')
    plt.show()
else:
    print("Unexpected JSON structure:", data)


<H1> Let's build a sales order specific prompt and function</H1>


In [1]:
import requests as r
USER_input= 'Can you tell me how many orders placed after 8th july 1996 have delayed deliveries ?'

In [2]:
text= \
f'''
Function:

def capture_date_time(year: int, month: int, day: int)
You need to capture the date and time in yyyy-mm-dd format

Example Usage:

capture_date_time(year=1996, month=7, day=8)



User Query: {USER_input}<human_end>

'''

In [3]:
from utils import query_raven
from datetime import datetime
call = query_raven(text)

In [4]:
print(call)

capture_date_time(year=1996, month=7, day=8)


In [5]:
def capture_date_time(year, month, day):
   # Format the date string with leading zeros
   formatted_date = datetime(year, month, day).strftime("%Y-%m-%dT00:00:00Z")
# Build your url
   url = f'https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter=RequiredDate%20ge%20{formatted_date}&$format=json'
   response = r.get(url)
   return response.json()
    

In [6]:
result = eval(call)
# print(result)

In [ ]:
# import os
# from dotenv import load_dotenv  
# from langchain_openai import ChatOpenAI

# # Load environment variables from .env file
# load_dotenv()
# # Access the API key using the variable name defined in the .env file
# api_key = os.getenv("OPENAI_API_KEY")
# model = ChatOpenAI(model="gpt-3.5-turbo")
# # model = ChatOpenAI(model="gpt-4")

In [ ]:
from langchain_community.llms import Ollama
model = Ollama(model="Gemma-27b")
# result = llm.invoke("Why is the sky blue?")
# print(result)

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
        #SystemMessage(content=f"query -->{USER_input}"),
    HumanMessage(content=f"Read this data . It is the complete dataset--> {result}+. and answer the following query --> tell me how many orders shipped after 8th july 1996 have reached after required date?"),
]


result=model.invoke(messages)
print(result)

In [ ]:
# print(messages)

<H1> Begin the POC Q&A Storyline </h1>


In [10]:
import accelerate
from langchain_community.llms import Ollama
model = Ollama(model="Gemma-27b")
# result = llm.invoke("Why is the sky blue?")
# print(result)

/home/avinash/anaconda3/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are an expert API URL Creator"),
    HumanMessage(content="Provide me an API to get order details from Northwind Odata API")
]

messages = [
    SystemMessage(content="You are an expert API URL Creator"),
    HumanMessage(content="Provide me an API to get order details from Northwind Odata API")
]

result=model.invoke(messages)
print(result)

```
https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$select=OrderID,CustomerID,EmployeeID,OrderDate,RequiredDate,ShippedDate,ShipVia,Freight,ShipName,ShipAddress,ShipCity,ShipRegion,ShipPostalCode,ShipCountry
```

This API URL will return a list of all orders from the Northwind OData service, including the following details for each order:

* **OrderID:** The unique ID of the order.
* **CustomerID:** The ID of the customer who placed the order.
* **EmployeeID:** The ID of the employee who took the order.
* **OrderDate:** The date the order was placed.
* **RequiredDate:** The date the order is required to be shipped by.
* **ShippedDate:** The date the order was shipped (null if not yet shipped).
* **ShipVia:** The ID of the shipper used for the order.
* **Freight:** The shipping cost of the order.
* **ShipName:** The name of the shipping recipient.
* **ShipAddress:** The address of the shipping recipient.
* **ShipCity:** The city of the shipping recipient.
* **ShipRegion:**

In [4]:
from utils import query_raven

USER_QUERY= 'Can you tell me ?'


raven_functions = \
f'''
def get_northwind_data_details(Category1 :str, Category2 :str, Category3 :str,Category4 :str):
    """

    
    The category variable can fall under any of these given below. Depending on the question asked you need to extaract relevant categories to be passed on to the API.

     **OrderID:** The unique ID of the order.
     **CustomerID:** The ID of the customer who placed the order.
     **EmployeeID:** The ID of the employee who took the order.
     **OrderDate:** The date the order was placed.
     **RequiredDate:** The date the order is required to be shipped by.
     **ShippedDate:** The date the order was shipped (null if not yet shipped).
     **ShipVia:** The ID of the shipper used for the order.
     **Freight:** The shipping cost of the order.
     **ShipName:** The name of the shipping recipient.
     **ShipAddress:** The address of the shipping recipient.
     **ShipCity:** The city of the shipping recipient.
     **ShipRegion:** The region of the shipping recipient (e.g., state or province).
     **ShipPostalCode:** The postal code of the shipping recipient.
     **ShipCountry:** The country of the shipping recipient.

    """

User Query: {USER_QUERY}<human_end>
'''
call = query_raven(raven_functions)
print(call)

get_northwind_data_details(Category1='OrderDate', Category2='ShippedDate', Category3='OrderID', Category4='ShipName')


<H1>make multiple function calls based on the number of parameters to be picked up <H1>

In [46]:
import requests as r
from utils import query_raven

In [47]:
def get_single_category_and_value(cat1 : str, value1 : str):

    url = f"https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter={cat1} eq {value1}&$format=json"
    response = r.get(url)
    return response.json()

def get_two_categories_and_values(cat1 : str, value1 : str, category2 : str, value2 : str):

    url = f"https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter={cat1} eq {value1} and {category2} eq {value2}&$format=json"
    response = r.get(url)
    return response.json()

def get_three_categories_and_values(cat1 : str, value1 : str, category2 : str, value2 : str, category3 : str, value3 : str):

    url = f"https://services.odata.org/V4/Northwind/Northwind.svc/Orders?$filter={cat1} eq {value1} and {category2} eq {value2} and {category3} eq {value3}&$format=json"
    response = r.get(url)
    return response.json()


In [53]:
USER_QUERY= 'give some insights regarding shipper with ID 3.'

In [54]:
raven_functions = \
f'''

Select the appropriate function depending on the number of parameters to be captured
""

def get_single_category_and_value(cat1 : str, value1 : str):

def get_two_categories_and_values(cat1 : str, value1 : str, category2 : str, value2 : str):

def get_three_categories_and_values(cat1 : str, value1 : str, category2 : str, value2 : str, category3 : str, value3 : str):


    The category variable can fall under any of these given below. Depending on the question asked you need to extaract relevant categories to be passed on to the API.

     **OrderID:** The unique ID of the order.
     **CustomerID:** The ID of the customer who placed the order.
     **EmployeeID:** The ID of the employee who took the order.
     **OrderDate:** The date the order was placed.
     **RequiredDate:** The date the order is required to be shipped by.
     **ShippedDate:** The date the order was shipped (null if not yet shipped).
     **ShipVia:** The ID of the shipper used for the order.
     **Freight:** The shipping cost of the order.
     **ShipName:** The name of the shipping recipient.
     **ShipAddress:** The address of the shipping recipient.
     **ShipCity:** The city of the shipping recipient.
     **ShipRegion:** The region of the shipping recipient (e.g., state or province).
     **ShipPostalCode:** The postal code of the shipping recipient.
     **ShipCountry:** The country of the shipping recipient.

    """
If any value is null, then dont add the value
User Query: {USER_QUERY}<human_end>


    """
'''

call = query_raven(raven_functions)
print(call)

get_single_category_and_value(cat1='ShipVia', value1='3')


In [55]:
response = eval(call)

In [56]:
import accelerate
from langchain_community.llms import Ollama
model = Ollama(model="Gemma-27b")
# result = llm.invoke("Why is the sky blue?")
# print(result)

In [57]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are a json data analysis expert. You can parse through multiple properties and its values within a json dataset and give meaningful insights from it. You work best when there is only snippets of data and not the whole dataset. Make assumptions where necessary, do the analysis which is asked and then state the assumptions in the end"),
    HumanMessage(content=f"Parse through this json data {response} and answer his is a User Query +{USER_QUERY}+ .")
]

result=model.invoke(messages)
print(result)

This looks like a snippet of data returned from an API, likely representing a list of orders filtered by the `ShipVia` field with a value of 3.  

Here's a breakdown:

* **Orders Data:** Each item in the list is an order object containing details like OrderID, CustomerID, ShipDate, Freight, and shipping address information.
* **@odata.nextLink:** This indicates that there are more orders matching the filter criteria (`ShipVia eq 3`) available. The link provided can be used to fetch the next set of results.

**Insights Regarding Shipper with ID 3:**

Based on this data alone, we can infer the following about shipper with ID 3:

* **They handle a significant volume of orders:** The returned list contains multiple orders assigned to this shipper.
* **They ship internationally:** Orders are going to various countries like USA, France, Poland and Finland.
* **They may specialize in smaller shipments:**  Some freight amounts are relatively low (e.g., $8.50, $19.26), suggesting they might han